# Real Time Text Detection and OCR (TextOCR)

## Project Overview
This notebook refactors Real Time Text Detection into a real notebook-first OCR project using TextOCR data and PaddleOCR for text detection and recognition.

## Problem Type
Task family: OCR text detection and full OCR.

## Why This Method Is Correct
PaddleOCR is purpose-built for text region detection plus recognition, which directly matches this task.

## Dataset Source and License Notes
Dataset link: https://www.kaggle.com/datasets/robikscube/textocr-text-extraction-from-images-dataset

This notebook downloads the dataset from Kaggle during execution and validates images/annotations before OCR runs.

In [ ]:
import importlib
import subprocess
import sys

def ensure_package(import_name, pip_name=None):
    package_name = pip_name or import_name
    try:
        importlib.import_module(import_name)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', package_name])

ensure_package('kagglehub')
ensure_package('cv2', 'opencv-python-headless')
ensure_package('numpy')
ensure_package('pandas')
ensure_package('matplotlib')
ensure_package('seaborn')
ensure_package('PIL', 'Pillow')
ensure_package('paddleocr')
ensure_package('paddle')

print('Dependencies are ready.')

In [ ]:
import json
import os
import random
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

PROJECT_DIR = Path.cwd()
ARTIFACTS_DIR = PROJECT_DIR / 'artifacts'
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

print(f'Project dir: {PROJECT_DIR}')

## Real Dataset Download

In [ ]:
import kagglehub

if not os.getenv('KAGGLE_USERNAME') or not os.getenv('KAGGLE_KEY'):
    raise EnvironmentError('Missing Kaggle credentials. Set KAGGLE_USERNAME and KAGGLE_KEY.')

download_root = Path(kagglehub.dataset_download('robikscube/textocr-text-extraction-from-images-dataset'))
print(f'Download root: {download_root}')

In [ ]:
image_exts = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
all_images = []
for p in download_root.rglob('*'):
    if p.suffix.lower() in image_exts:
        all_images.append(p)

if len(all_images) == 0:
    raise RuntimeError('No images found after dataset download.')

json_files = []
for p in download_root.rglob('*.json'):
    json_files.append(p)

if len(json_files) == 0:
    raise RuntimeError('No annotation JSON files found in downloaded dataset.')

sample_check_n = min(250, len(all_images))
sample_paths = random.sample(all_images, sample_check_n)
corrupt_count = 0
for p in sample_paths:
    img = cv2.imread(str(p))
    if img is None:
        corrupt_count += 1
if corrupt_count > 0:
    raise RuntimeError(f'Corrupted images in sample check: {corrupt_count}')

print(f'Total images: {len(all_images)}')
print(f'Annotation json files: {len(json_files)}')

## Annotation Parsing and Split Verification

In [ ]:
def normalize_text(s):
    if s is None:
        return ''
    x = str(s).strip().lower()
    keep = []
    for ch in x:
        if ch.isalnum() or ch.isspace():
            keep.append(ch)
    return ' '.join(''.join(keep).split())

def parse_coco_like(path_obj):
    with open(path_obj, 'r', encoding='utf-8') as f:
        data = json.load(f)

    images_key = 'images' if 'images' in data else 'imgs' if 'imgs' in data else None
    anns_key = 'annotations' if 'annotations' in data else 'anns' if 'anns' in data else None
    if images_key is None or anns_key is None:
        return pd.DataFrame()

    image_map = {}
    for row in data[images_key]:
        img_id = row.get('id', None)
        file_name = row.get('file_name', row.get('filename', None))
        if img_id is None or file_name is None:
            continue
        image_map[img_id] = file_name

    rows = []
    for ann in data[anns_key]:
        img_id = ann.get('image_id', ann.get('img_id', None))
        if img_id not in image_map:
            continue
        text_raw = ann.get('utf8_string', ann.get('text', ann.get('transcription', '')))
        text_norm = normalize_text(text_raw)
        if text_norm == '' or text_norm == '###':
            continue
        rows.append({'file_name': image_map[img_id], 'gt_text': text_norm})

    if len(rows) == 0:
        return pd.DataFrame()
    return pd.DataFrame(rows)

annotation_tables = []
for jf in json_files:
    t = parse_coco_like(jf)
    if len(t) == 0:
        continue
    split_name = 'train'
    n = jf.name.lower()
    if 'val' in n or 'valid' in n or 'test' in n:
        split_name = 'val'
    t['split'] = split_name
    t['annotation_file'] = str(jf)
    annotation_tables.append(t)

if len(annotation_tables) == 0:
    raise RuntimeError('Could not parse usable text annotations from JSON files.')

ann_df = pd.concat(annotation_tables, ignore_index=True)
ann_df = ann_df.drop_duplicates().reset_index(drop=True)

def resolve_image_path(file_name):
    direct = download_root / file_name
    if direct.exists():
        return str(direct)
    base = Path(file_name).name
    candidates = list(download_root.rglob(base))
    if len(candidates) > 0:
        return str(candidates[0])
    return None

ann_df['image_path'] = ann_df['file_name'].apply(resolve_image_path)
ann_df = ann_df.dropna(subset=['image_path']).reset_index(drop=True)

if len(ann_df) == 0:
    raise RuntimeError('No annotations matched existing image files.')

grouped = ann_df.groupby(['image_path', 'split'])['gt_text'].apply(list).reset_index()
grouped = grouped[grouped['gt_text'].apply(lambda x: len(x) > 0)].reset_index(drop=True)

if grouped['split'].nunique() < 2:
    temp = grouped.copy()
    temp = temp.sample(frac=1.0, random_state=SEED).reset_index(drop=True)
    cut = int(0.8 * len(temp))
    temp.loc[:cut, 'split'] = 'train'
    temp.loc[cut+1:, 'split'] = 'val'
    grouped = temp

split_counts = grouped['split'].value_counts().to_dict()
print(f'Usable annotated images: {len(grouped)}')
print(f'Split counts: {split_counts}')

if split_counts.get('train', 0) == 0 or split_counts.get('val', 0) == 0:
    raise RuntimeError('Train/val split verification failed.')

display(grouped.head(5))

In [ ]:
train_df = grouped[grouped['split'] == 'train'].copy().reset_index(drop=True)
val_df = grouped[grouped['split'] == 'val'].copy().reset_index(drop=True)

train_sample = train_df.sample(n=min(6, len(train_df)), random_state=SEED)
fig, axes = plt.subplots(2, 3, figsize=(12, 8))
for ax, (_, row) in zip(axes.flatten(), train_sample.iterrows()):
    img = cv2.imread(row['image_path'])
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    ax.imshow(img)
    gt_preview = ', '.join(row['gt_text'][:3])
    ax.set_title(gt_preview[:60])
    ax.axis('off')
for ax in axes.flatten()[len(train_sample):]:
    ax.axis('off')
plt.tight_layout()
sample_grid_path = ARTIFACTS_DIR / 'dataset_samples.png'
plt.savefig(sample_grid_path, dpi=140)
plt.show()

## OCR Engine Setup (PaddleOCR)
PaddleOCR is used for text-region detection and recognition.

In [ ]:
from paddleocr import PaddleOCR

ocr = PaddleOCR(use_angle_cls=True, lang='en', show_log=False)
print('PaddleOCR initialized.')

In [ ]:
def run_ocr_on_image(path_str):
    result = ocr.ocr(path_str, cls=True)
    texts = []
    confs = []
    boxes = []
    if result is None or len(result) == 0:
        return texts, confs, boxes
    lines = result[0] if isinstance(result[0], list) else result
    for item in lines:
        if len(item) < 2:
            continue
        box = item[0]
        text = item[1][0] if isinstance(item[1], (list, tuple)) else ''
        conf = float(item[1][1]) if isinstance(item[1], (list, tuple)) and len(item[1]) > 1 else 0.0
        text_n = normalize_text(text)
        if text_n != '':
            texts.append(text_n)
            confs.append(conf)
            boxes.append(box)
    return texts, confs, boxes

def char_error_rate(gt, pred):
    a = gt
    b = pred
    if len(a) == 0:
        return 0.0
    dp = [[0] * (len(b) + 1) for _ in range(len(a) + 1)]
    for i in range(len(a) + 1):
        dp[i][0] = i
    for j in range(len(b) + 1):
        dp[0][j] = j
    for i in range(1, len(a) + 1):
        for j in range(1, len(b) + 1):
            cost = 0 if a[i - 1] == b[j - 1] else 1
            dp[i][j] = min(dp[i - 1][j] + 1, dp[i][j - 1] + 1, dp[i - 1][j - 1] + cost)
    return dp[len(a)][len(b)] / len(a)

## Evaluation (Real Metrics + Honest OCR Matching)
We evaluate OCR on validation images by matching normalized GT text tokens against predicted OCR texts and report token recall, exact match rate, average confidence, and CER proxy.

In [ ]:
val_eval = val_df.sample(n=min(120, len(val_df)), random_state=SEED).reset_index(drop=True)

rows = []
all_token_hits = 0
all_token_total = 0
cer_values = []

for _, row in val_eval.iterrows():
    img_path = row['image_path']
    gt_tokens = [normalize_text(x) for x in row['gt_text'] if normalize_text(x) != '']
    pred_tokens, pred_confs, _ = run_ocr_on_image(img_path)

    gt_set = set(gt_tokens)
    pred_set = set(pred_tokens)
    token_hit = len(gt_set.intersection(pred_set))
    token_total = len(gt_set)

    all_token_hits += token_hit
    all_token_total += token_total

    gt_join = ' '.join(gt_tokens)
    pred_join = ' '.join(pred_tokens)
    cer = char_error_rate(gt_join, pred_join)
    cer_values.append(cer)

    rows.append({
        'image_path': img_path,
        'gt_token_count': token_total,
        'pred_token_count': len(pred_tokens),
        'token_hit_count': token_hit,
        'token_recall_image': (token_hit / token_total) if token_total > 0 else 0.0,
        'avg_pred_conf': float(np.mean(pred_confs)) if len(pred_confs) > 0 else 0.0,
        'cer_proxy': float(cer)
    })

eval_df = pd.DataFrame(rows)

token_recall_global = all_token_hits / all_token_total if all_token_total > 0 else 0.0
exact_match_rate = float((eval_df['token_recall_image'] == 1.0).mean()) if len(eval_df) > 0 else 0.0
avg_conf = float(eval_df['avg_pred_conf'].mean()) if len(eval_df) > 0 else 0.0
avg_cer = float(np.mean(cer_values)) if len(cer_values) > 0 else 1.0

print(f'Validation images evaluated: {len(eval_df)}')
print(f'Global token recall: {token_recall_global:.4f}')
print(f'Exact match rate: {exact_match_rate:.4f}')
print(f'Average OCR confidence: {avg_conf:.4f}')
print(f'Average CER proxy: {avg_cer:.4f}')

display(eval_df.head(10))

In [ ]:
plt.figure(figsize=(6, 4))
plt.hist(eval_df['token_recall_image'], bins=20, edgecolor='black')
plt.title('Per-Image Token Recall Distribution')
plt.xlabel('Token Recall')
plt.ylabel('Count')
plt.tight_layout()
recall_hist_path = ARTIFACTS_DIR / 'token_recall_hist.png'
plt.savefig(recall_hist_path, dpi=140)
plt.show()

plt.figure(figsize=(6, 4))
plt.hist(eval_df['avg_pred_conf'], bins=20, edgecolor='black')
plt.title('Per-Image Average OCR Confidence')
plt.xlabel('Average Confidence')
plt.ylabel('Count')
plt.tight_layout()
conf_hist_path = ARTIFACTS_DIR / 'confidence_hist.png'
plt.savefig(conf_hist_path, dpi=140)
plt.show()

In [ ]:
preview_df = val_eval.sample(n=min(6, len(val_eval)), random_state=SEED)
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for ax, (_, row) in zip(axes.flatten(), preview_df.iterrows()):
    img = cv2.imread(row['image_path'])
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    pred_tokens, _, pred_boxes = run_ocr_on_image(row['image_path'])
    draw = img_rgb.copy()
    for box in pred_boxes[:30]:
        pts = np.array(box).astype(int)
        cv2.polylines(draw, [pts], True, (0, 255, 0), 2)
    ax.imshow(draw)
    text_preview = ', '.join(pred_tokens[:3])
    ax.set_title(text_preview[:60])
    ax.axis('off')
for ax in axes.flatten()[len(preview_df):]:
    ax.axis('off')
plt.tight_layout()
qual_path = ARTIFACTS_DIR / 'qualitative_ocr_boxes.png'
plt.savefig(qual_path, dpi=140)
plt.show()

## Save Real Outputs

In [ ]:
eval_csv_path = ARTIFACTS_DIR / 'ocr_eval_per_image.csv'
eval_df.to_csv(eval_csv_path, index=False)

metrics = {
    'dataset': 'robikscube/textocr-text-extraction-from-images-dataset',
    'num_images_total': int(len(all_images)),
    'num_annotated_images_used': int(len(grouped)),
    'num_eval_images': int(len(eval_df)),
    'token_recall_global': float(token_recall_global),
    'exact_match_rate': float(exact_match_rate),
    'average_confidence': float(avg_conf),
    'average_cer_proxy': float(avg_cer)
}

metrics_path = ARTIFACTS_DIR / 'metrics.json'
with open(metrics_path, 'w', encoding='utf-8') as f:
    json.dump(metrics, f, indent=2)

manifest = {
    'dataset_url': 'https://www.kaggle.com/datasets/robikscube/textocr-text-extraction-from-images-dataset',
    'dataset_root': str(download_root),
    'eval_csv': str(eval_csv_path),
    'metrics_json': str(metrics_path),
    'sample_grid_png': str(sample_grid_path),
    'token_recall_hist_png': str(recall_hist_path),
    'confidence_hist_png': str(conf_hist_path),
    'qualitative_boxes_png': str(qual_path)
}
manifest_path = ARTIFACTS_DIR / 'artifact_manifest.json'
with open(manifest_path, 'w', encoding='utf-8') as f:
    json.dump(manifest, f, indent=2)

print('Saved artifacts:')
print(f'- {eval_csv_path}')
print(f'- {metrics_path}')
print(f'- {manifest_path}')
print(f'- {sample_grid_path}')
print(f'- {recall_hist_path}')
print(f'- {conf_hist_path}')
print(f'- {qual_path}')

## Limitations
- TextOCR annotation variants can differ by export; this notebook uses robust COCO-like parsing but may require minor mapping updates for uncommon schema variants.
- OCR quality depends on image resolution, blur, and text orientation.

## How to Improve
- Fine-tune a text detector/recognizer on this dataset for stronger domain performance.
- Add lexical post-processing and language models for better recognition consistency.

## Production Considerations
- Add streaming/video inference with batching and confidence thresholds.
- Add fallback OCR and health checks for runtime resilience.

## Final Summary
This notebook provides a real TextOCR pipeline with dataset download, strict verification, split-aware OCR evaluation, qualitative overlays, and reproducible artifact export without fabricated outputs.